# 05 — Final model export & dashboard data

Run **after Notebook 04**. This notebook uses its saved model; it does not tune or retrain. It exports metrics, charts and model details for FastAPI. The test set is used only to reproduce the already-finalized evaluation. Run from the project root.

In [ ]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve

DATA = Path("data/processed")
OUT = Path("artifacts"); OUT.mkdir(exist_ok=True)
model = joblib.load("models/credit_risk_pipeline.joblib")
train = pd.read_parquet(DATA / "train_fe.parquet")
test = pd.read_parquet(DATA / "test_fe.parquet")
FEATURES = ["person_age", "person_income", "person_emp_length", "loan_amnt", "cb_person_cred_hist_length", "loan_to_income_ratio", "person_home_ownership", "loan_intent", "cb_person_default_on_file"]
y_test = test["loan_status"]
p = model.predict_proba(test[FEATURES])[:, 1]
y_pred = (p >= 0.5).astype(int)

def save(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


## 1. Final evaluation
The holdout metrics below are descriptive results, not a new opportunity to tune hyperparameters.

In [ ]:
matrix = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
metrics = {
    "split": "holdout_test", "threshold": 0.5,
    "roc_auc": float(roc_auc_score(y_test, p)),
    "average_precision": float(average_precision_score(y_test, p)),
    "precision_default": float(precision_score(y_test, y_pred)),
    "recall_default": float(recall_score(y_test, y_pred)),
    "f1_default": float(f1_score(y_test, y_pred)),
    "confusion_matrix": matrix,
    "support": int(len(y_test)),
}
save("metrics.json", metrics)
display(pd.Series(metrics))

## 2. Information for the model page

In [ ]:
parameters = model.named_steps["model"].get_params()
parameters = {k: parameters[k] for k in ["n_estimators", "min_samples_leaf", "max_features", "max_depth"]}
info = {
    "name": "Tuned Random Forest", "version": "tuned-rf-v1",
    "target": "loan_status", "default_threshold": 0.5,
    "positive_class": 1, "model_status": "demo_only_not_credit_approval",
    "features": FEATURES,
    "parameters": parameters,
    "cv_average_precision": 0.7947,  # Replace if your saved tuning result differs.
    "feature_policy": "application_time_without_loan_grade_or_interest_rate",
}
save("model_info.json", info)


## 3. Dashboard statistics (training data only)

In [ ]:
overview = {
    "split": "train", "observations": int(len(train)),
    "default_count": int((train.loan_status == 1).sum()),
    "nondefault_count": int((train.loan_status == 0).sum()),
    "default_rate": float(train.loan_status.mean()),
}
save("overview.json", overview)

dists = {}
for feature in ["person_income", "loan_amnt", "person_age", "loan_to_income_ratio"]:
    v = train[feature].dropna().to_numpy()
    counts, edges = np.histogram(v, bins=15)
    dists[feature] = [{"start": float(edges[i]), "end": float(edges[i+1]), "count": int(counts[i])} for i in range(len(counts))]
save("distributions.json", {"split":"train", "histograms": dists})

rates = {}
for feature in ["person_home_ownership", "loan_intent", "cb_person_default_on_file"]:
    grouped = train.groupby(feature, dropna=False).loan_status.agg(["mean", "count"]).reset_index()
    rates[feature] = [{"category":str(r[feature]), "default_rate":float(r["mean"]), "count":int(r["count"])} for _,r in grouped.iterrows()]
save("default_rates.json", {"split": "train", "groups": rates})


## 4. Evaluation curves
Curves are from the single held-out evaluation. Show these as historical diagnostics, not real-time model performance.

In [ ]:
fpr, tpr, rt = roc_curve(y_test, p)
precision, recall, pt = precision_recall_curve(y_test, p)
# Thin to at most ~250 points per curve for browser rendering.
def thin(*arrays, limit=250):
    ix = np.unique(np.linspace(0, len(arrays[0])-1, min(limit, len(arrays[0])), dtype=int))
    return [[float(v) for v in a[ix]] for a in arrays]
fpr_v, tpr_v = thin(fpr, tpr)
precision_v, recall_v = thin(precision, recall)
save("curves.json", {
    "split": "holdout_test",
    "roc": [{"fpr":a, "tpr":b} for a,b in zip(fpr_v,tpr_v)],
    "precision_recall": [{"recall":a, "precision":b} for a,b in zip(recall_v,precision_v)],
})
print("Exported 6 JSON files to artifacts/")


## 5. Next commands
From the project root:
```bash
uvicorn api.main:app --reload
# second terminal
mlflow server --host 127.0.0.1 --port 5000
# third terminal
python -m scripts.track_mlflow
```
The serialized model expects engineered features, so the FastAPI endpoint applies `engineer_features()` before calling it. The same feature function is copied into `src/features.py` after development.